# Google Colab T4 Training Workflow

This notebook retrains the classical baselines and RoBERTa on a Google Colab T4 GPU runtime, then regenerates the richer evaluation reports and figures.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/LOKESHBOOTU/SENTIMENT-ANALYSIS-ON-AMAZON-REVIEWS.git"
REPO_DIR = Path("/content/SENTIMENT-ANALYSIS-ON-AMAZON-REVIEWS")

if not REPO_DIR.exists():
    !git clone {REPO_URL}

%cd /content/SENTIMENT-ANALYSIS-ON-AMAZON-REVIEWS
!pip install -q -r requirements.txt

In [ ]:
import platform
import torch

print("Python:", platform.python_version())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!python -m src.preprocessing
!python -m src.train_ml --force-retrain --n-jobs 2

In [ ]:
!python -m src.train_roberta --colab-t4-profile --force-retrain --selection-metric accuracy

In [ ]:
import json
import pandas as pd
from pathlib import Path

ROOT = Path("/content/SENTIMENT-ANALYSIS-ON-AMAZON-REVIEWS")
REPORTS_DIR = ROOT / "outputs" / "reports"

display(pd.read_csv(REPORTS_DIR / "ml_model_comparison.csv"))
display(pd.read_csv(REPORTS_DIR / "roberta_trials.csv").head(10))

with (REPORTS_DIR / "roberta_summary.json").open("r", encoding="utf-8") as file:
    summary = json.load(file)

print(json.dumps(summary, indent=2))

In [ ]:
from IPython.display import Image, display
from pathlib import Path

FIGURES_DIR = ROOT / "outputs" / "figures"
BEST_MODEL_DIR = Path(summary["best_model_dir"])

figure_paths = [
    FIGURES_DIR / "all_model_accuracy_comparison.png",
    FIGURES_DIR / "all_model_metric_dashboard.png",
    FIGURES_DIR / "roberta_confusion_matrix.png",
    FIGURES_DIR / "roberta_normalized_confusion_matrix.png",
    FIGURES_DIR / "roberta_roc_curves.png",
    FIGURES_DIR / "roberta_precision_recall_curves.png",
    FIGURES_DIR / "roberta_confidence_analysis.png",
    BEST_MODEL_DIR / "training_validation_curves.png",
    BEST_MODEL_DIR / "learning_rate_schedule.png",
]

for figure_path in figure_paths:
    if figure_path.exists():
        display(Image(filename=str(figure_path)))